In [1]:
# ============================================
# UAE Mobile Intelligence - Ookla Processing
# Quarter: 2026 Q2
# ============================================

import duckdb
import pandas as pd
import geopandas as gpd

In [2]:
# --------------------------------------------
# 1. File paths
# --------------------------------------------

ookla_file = "../data/raw/ookla/2026-04-01_performance_mobile_tiles.parquet"

boundary_file = "../data/raw/boundary/uae_boundary.geojson"


In [6]:
# --------------------------------------------
# 3. Rough UAE geographic pre-filter
# --------------------------------------------
# IMPORTANT:
# This is NOT the final UAE filter.
# It only reduces the global dataset before
# the exact UAE polygon check.

uae_candidate = duckdb.sql(f"""
    SELECT *
    FROM read_parquet('{ookla_file}')
    WHERE tile_x BETWEEN 51 AND 57
      AND tile_y BETWEEN 22 AND 27
""").df()

print("Rows after rough bounding-box filter:", len(uae_candidate))

display(
    uae_candidate[
        [
            "tile_x",
            "tile_y",
            "avg_d_kbps",
            "avg_u_kbps",
            "avg_lat_ms",
            "tests",
            "devices"
        ]
    ].head(10)
)


Rows after rough bounding-box filter: 10367


,tile_x,tile_y,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests,devices
0,51.1935,26.1283,684606,45479,16,1,1
1,51.1880,26.1234,216590,34538,23,2,2
2,51.2045,26.1382,588652,22895,18,1,1
3,51.2100,26.1431,406596,16433,27,2,1
4,51.2155,26.1382,597082,55377,8,2,2
5,51.1990,26.1332,18008,8233,26,1,1
6,51.2045,26.1332,261285,18333,21,4,1
7,51.1990,26.1283,43481,4464,19,1,1
8,51.2045,26.1283,270954,24125,20,4,3
9,51.2155,26.1185,840404,59175,21,1,1


In [7]:
# --------------------------------------------
# 4. Load the real UAE boundary
# --------------------------------------------

uae_boundary = gpd.read_file(boundary_file)

print("Original boundary CRS:", uae_boundary.crs)
print("Boundary geometry type:")
print(uae_boundary.geometry.geom_type)


# Convert boundary explicitly to EPSG:4326
uae_boundary = uae_boundary.to_crs("EPSG:4326")

print("Boundary CRS after conversion:", uae_boundary.crs)

Original boundary CRS: EPSG:4326
Boundary geometry type:
0    MultiPolygon
dtype: str
Boundary CRS after conversion: EPSG:4326


In [8]:
# --------------------------------------------
# 5. Convert Ookla tile centroids to points
# --------------------------------------------
# Ookla:
# tile_x = longitude
# tile_y = latitude

candidate_points = gpd.GeoDataFrame(
    uae_candidate,
    geometry=gpd.points_from_xy(
        uae_candidate["tile_x"],
        uae_candidate["tile_y"]
    ),
    crs="EPSG:4326"
)

print("Candidate points CRS:", candidate_points.crs)

display(
    candidate_points[
        ["tile_x", "tile_y", "geometry"]
    ].head()
)

Candidate points CRS: EPSG:4326


,tile_x,tile_y,geometry
0,51.1935,26.1283,POINT (51.1935 26.1283)
1,51.1880,26.1234,POINT (51.188 26.1234)
2,51.2045,26.1382,POINT (51.2045 26.1382)
3,51.2100,26.1431,POINT (51.21 26.1431)
4,51.2155,26.1382,POINT (51.2155 26.1382)


In [9]:
# --------------------------------------------
# 6. Exact UAE point-in-polygon filter
# --------------------------------------------

uae_tiles = gpd.sjoin(
    candidate_points,
    uae_boundary[["geometry"]],
    predicate="within",
    how="inner"
)

print("Rows before exact UAE boundary:", len(candidate_points))
print("Rows after exact UAE boundary:", len(uae_tiles))

Rows before exact UAE boundary: 10367
Rows after exact UAE boundary: 6879


In [10]:
# --------------------------------------------
# 7. Inspect final UAE tiles
# --------------------------------------------

display(
    uae_tiles[
        [
            "quadkey",
            "tile_x",
            "tile_y",
            "avg_d_kbps",
            "avg_u_kbps",
            "avg_lat_ms",
            "avg_lat_down_ms",
            "avg_lat_up_ms",
            "tests",
            "devices"
        ]
    ].head(10)
)

,quadkey,tile_x,tile_y,avg_d_kbps,avg_u_kbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,tests,devices
1844,1230231133031333,56.0715,26.0000,8078,2390,137,892.0,2633.0,1,1
1845,1230231133033132,56.0660,25.9803,178806,7675,14,223.0,875.0,1,1
1846,1230231133033133,56.0715,25.9803,590386,72993,19,490.0,182.0,1,1
1847,1230231133033321,56.0605,25.9655,561968,25099,17,322.0,1833.0,2,1
1848,1230231133033322,56.0550,25.9605,110759,23215,280,1059.0,611.0,1,1
1850,1230231133102210,56.0880,26.0543,550697,92707,30,417.0,168.0,1,1
1851,1230231133102232,56.0880,26.0395,66445,9143,19,276.0,954.0,3,2
1853,1230231133120012,56.0880,26.0296,493102,51929,13,220.0,231.0,2,1
1854,1230231133120021,56.0825,26.0247,1330651,32298,14,228.0,234.0,3,1
1855,1230231133120030,56.0880,26.0247,736661,37986,16,429.0,662.0,4,3


In [11]:
# --------------------------------------------
# 8. Basic validation
# --------------------------------------------

print("UAE tile count:", len(uae_tiles))
print("Total tests:", uae_tiles["tests"].sum())
print("Summed tile device counts:", uae_tiles["devices"].sum())

UAE tile count: 6879
Total tests: 41223
Summed tile device counts: 22958


In [12]:
# --------------------------------------------
# 9. Convert speed units to Mbps
# --------------------------------------------

uae_tiles["download_mbps"] = uae_tiles["avg_d_kbps"] / 1000
uae_tiles["upload_mbps"] = uae_tiles["avg_u_kbps"] / 1000


In [13]:
# --------------------------------------------
# 10. Basic descriptive statistics
# --------------------------------------------

display(
    uae_tiles[
        [
            "download_mbps",
            "upload_mbps",
            "avg_lat_ms",
            "avg_lat_down_ms",
            "avg_lat_up_ms",
            "tests",
            "devices"
        ]
    ].describe()
)

,download_mbps,upload_mbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,tests,devices
count,6879.000000,6879.000000,6879.000000,6819.000000,6843.000000,6879.000000,6879.000000
mean,414.551638,31.237349,30.390173,610.744831,973.617273,5.992586,3.337404
std,361.460965,27.103569,38.112555,716.147802,970.591322,14.504537,8.164917
min,0.050000,0.002000,0.000000,21.000000,19.000000,1.000000,1.000000
25%,117.143500,11.477500,16.000000,300.000000,332.000000,1.000000,1.000000
50%,335.930000,25.053000,21.000000,404.000000,648.000000,2.000000,2.000000
75%,612.374500,42.702500,29.000000,617.500000,1257.500000,5.000000,3.000000
max,2193.000000,325.974000,684.000000,9907.000000,9619.000000,458.000000,324.000000
